# Tempo Run 2025 — Enrich data bằng Qwen3 API

**Tuỳ chọn**: Bỏ qua nếu không muốn dùng API hoặc đã hết quota.
Mặc định chỉ paraphrase (giữ đáp án gốc). Dùng `--synth` để sinh thêm câu hỏi mới (tốn thêm token).

**Cảnh báo quota**: 1M token free. Paraphrase ~3000 mẫu có thể tốn ~600K-1M token. Bắt đầu với `--synth-max 0` để tiết kiệm.

In [ ]:
%cd /content/finetune_1B_MCQ_VN
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")
import os
assert os.environ.get("DASHSCOPE_API_KEY"), "DASHSCOPE_API_KEY missing in .env"

In [ ]:
# 1. Smoke test API (1 call)
from temprun.enrich import call_chat, get_client, get_model
client = get_client()
model = get_model()
out = call_chat(client, [
    {"role": "system", "content": "Bạn là trợ lý hữu ích."},
    {"role": "user", "content": "Trả lời ngắn gọn: 2+2=?"},
], max_tokens=20, temperature=0.0)
print('model:', model)
print('output:', out)

In [ ]:
# 2. Chạy enrichment thực sự (idempotent — chạy lại không tốn thêm token cho row đã xử lý)
# Mặc định: paraphrase + explain. Không sinh Q mới.
!python scripts/enrich_data.py \
    --in  data/processed/train.jsonl \
    --out data/processed/enriched.jsonl

In [ ]:
# 3. (Tuỳ chọn) Thêm sinh câu hỏi mới — tốn thêm ~200K token
# Chỉ chạy nếu còn quota. Bỏ comment nếu muốn.

# !python scripts/enrich_data.py \
#     --in  data/processed/enriched.jsonl \
#     --out data/processed/enriched.jsonl \
#     --no-paraphrase --no-explain --synth --synth-max 200

In [ ]:
# 4. Merge enriched → final/{train,eval}.jsonl (90/10 split lại)
!python scripts/merge_enriched.py

In [ ]:
# 5. Verify final size
!wc -l data/processed/final/*.jsonl
!head -1 data/processed/final/train.jsonl | python3 -c "import json,sys; r=json.loads(sys.stdin.read()); print('label:', r['label']); print('source:', r.get('_source', 'original'))"